In [ ]:
import pandas as pd
from google.colab import files

df_train=pd.read_csv("/kaggle/input/competitions/triagegeist/train.csv")
df_test=pd.read_csv("/kaggle/input/competitions/triagegeist/test.csv")
df_train.head()

vitals_columns=[
    "systolic_bp",
    "diastolic_bp",
    "respiratory_rate",
    "temperature_c"

]
for col in vitals_columns:
    df_train[col+"_missing"] = df_train[col].isnull().astype(int)
    df_train[col] = df_train[col].fillna(df_train[col].median())

for col in vitals_columns:
    df_test[col + "_missing"] = df_test[col].isnull().astype(int)
    df_test[col] = df_test[col].fillna(df_train[col].median())



# Recalculate derived features
df_train['mean_arterial_pressure'] = (
    2 * df_train['diastolic_bp'] + df_train['systolic_bp']
) / 3

df_train['pulse_pressure'] = (
    df_train['systolic_bp'] - df_train['diastolic_bp']
)
df_test['mean_arterial_pressure'] = (
    2 * df_test['diastolic_bp'] + df_test['systolic_bp']
) / 3

df_test['pulse_pressure'] = (
    df_test['systolic_bp'] - df_test['diastolic_bp']
)

# Fill if any remain missing
derived_cols = ['mean_arterial_pressure', 'pulse_pressure']

for col in derived_cols:
    df_train[col] = df_train[col].fillna(df_train[col].median())

for col in ['mean_arterial_pressure', 'pulse_pressure']:
    df_test[col] = df_test[col].fillna(df_train[col].median())


df_train['shock_index'] = (
    df_train['heart_rate'] / df_train['systolic_bp']
)
df_train['shock_index'] = df_train['shock_index'].fillna(
    df_train['shock_index'].median()
)

df_test['shock_index'] = (
    df_test['heart_rate'] / df_test['systolic_bp']
)

df_test['shock_index'] = df_test['shock_index'].fillna(
    df_train['shock_index'].median()
)

df_train.isnull().sum()

import numpy as np

df_train['pain_score'] = df_train['pain_score'].replace(-1, np.nan)
df_train['pain_score'] = df_train['pain_score'].fillna(
    df_train['pain_score'].median()
)

df_test['pain_score'] = df_test['pain_score'].replace(-1, np.nan)

df_test['pain_score'] = df_test['pain_score'].fillna(
    df_train['pain_score'].median()
)

drop_cols = [
    "patient_id",
    "triage_nurse_id",
    "site_id",
    "disposition",
    "triage_acuity"
]
drop_cols_test = [
    "patient_id",
    "triage_nurse_id",
    "site_id",
    "disposition"
]


X_tra = df_train.drop(columns=drop_cols, errors="ignore")

y=df_train["triage_acuity"]

X_tes = df_test.drop(columns=drop_cols_test, errors="ignore")

X = pd.get_dummies(X_tra, drop_first=True)
X_test = pd.get_dummies(X_tes, drop_first=True)




# align columns
X_test = X_test.reindex(columns=X.columns, fill_value=0)

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

from sklearn.ensemble import RandomForestClassifier
R_model=RandomForestClassifier(    n_estimators=300,
    max_depth=15,
    class_weight="balanced",
    random_state=42)
model=R_model.fit(X_train,y_train)

from sklearn.metrics import accuracy_score,classification_report
y_pred=model.predict(X_val)
acs=accuracy_score(y_val,y_pred)
print ("accuracy score:", acs)
print(classification_report(y_val,y_pred))

model.fit(X, y)

test_pred = model.predict(X_test)


submission = pd.DataFrame({
    "patient_id": df_test["patient_id"],
    "triage_acuity": test_pred
})
submission.to_csv("submission.csv", index=False)



In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))